## Homework 3: Classification

This is the personal implementation of the homework 3 of [ML Zoomcamp 2026](https://github.com/DataTalksClub/machine-learning-zoomcamp).

In [1]:
# get the data
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-09-22 19:02:27--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.005s  

2026-09-22 19:02:27 (53.9 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



In [2]:
# prepare the dataset
import pandas as pd

df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# check missing values
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
# define categorical and numerical features
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

# fill missing categorical values with 'NA'
df[categorical] = df[categorical].fillna('NA')

# fill missing numerical values with 0.0
df[numerical] = df[numerical].fillna(0.0)

# check again
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

#### Q1. What is the most frequent observation (mode) for the column `industry`?

In [5]:
# most frequrent
print(f"The most frequent observation (mode): {df.industry.mode()[0]}")

# observation counts
df.industry.value_counts()

The most frequent observation (mode): technology


industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

#### Q2. What are the two features that have the biggest correlation?

In [6]:
# correlation matrix for numerical features
correlation_matrix = df[numerical].corr()

print(
    f"Highest correlation: interaction_count & lead_score "
    f"-> {correlation_matrix.loc['interaction_count', 'lead_score']:.3f}"
)
correlation_matrix

Highest correlation: interaction_count & lead_score -> 0.916


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [7]:
from sklearn.model_selection import train_test_split

# split the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# reset index for each
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# separate target variable
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

# delete target variable data from features
del df_train['converted']
del df_val['converted']
del df_test['converted']

# get the size of each set
print("Total size:", len(df))
print(f"Train, Val, Test: ({len(df_train)}, {len(df_val)}, {len(df_test)})")
print(f"y_train, y_val, y_test: ({len(y_train)}, {len(y_val)}, {len(y_test)})")

Total size: 5000
Train, Val, Test: (3000, 1000, 1000)
y_train, y_val, y_test: (3000, 1000, 1000)


#### Q3.Which of these variables has the biggest mutual information score?

In [8]:
from sklearn.metrics import mutual_info_score

# check for multiple columns
def mutual_info_churn_score(series):
    return round(mutual_info_score(series, y_train), 2)

# get and sort by their importance
mi = df_train[categorical].apply(mutual_info_churn_score)
print("The variable with the biggest mutual information score:", mi.sort_values(ascending=False).index[0])
mi.sort_values(ascending=False)

The variable with the biggest mutual information score: lead_source


lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

#### Q4. What accuracy did you get?

In [9]:
from sklearn.feature_extraction import DictVectorizer

# create a dict vectorizer (do not use sparse matrices)
dv = DictVectorizer(sparse=False)

# fit data to vectorizer and transform into one-hot-encoded (dict vectorizer does this only for categorical data)
train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

# check features after one-hot encoding
dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [16]:
# train logistic regression model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

# fit trains the model from scratch by fitting to training data
model.fit(X_train, y_train)

# predicted class: 0 = not converted, 1 = converted
y_pred_class = model.predict(X_val)

# predicted probabilities
y_pred_proba = model.predict_proba(X_val)

print("\nProbability columns:")
print(model.classes_)   # usually [0, 1]

# probability of converted = 1 only
y_pred = y_pred_proba[:, 1]
print("\nFirst 10 probability of converted predictions:")
print(y_pred[:10])

# classify customers as converted if predicted probability is at least 50%
is_converted = (y_pred >= 0.5)

# calculate how often the predicted converted matches the actual value
accuracy = (y_val == is_converted).mean()

print(f"\nPrediction accuracy: {accuracy:.2f}")
print(f"Percentage of correct predictions: {accuracy:.2%}")


Probability columns:
[0 1]

First 10 probability of converted predictions:
[0.61725691 0.44364108 0.60433996 0.64562246 0.52214159 0.77885722
 0.60698076 0.54481903 0.76770971 0.52765495]

Prediction accuracy: 0.65
Percentage of correct predictions: 64.50%


#### Q5. Which of following feature has the smallest difference?

In [17]:
excluded = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

features = categorical + numerical

for e in excluded:
    
    current = [
        f for f in features
        if f != e
    ]

    # create a dict vectorizer (do not use sparse matrices)
    dv = DictVectorizer(sparse=False)

    # fit data to vectorizer and transform into one-hot-encoded
    train_dict = df_train[current].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[current].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

    # fit trains the model from scratch by fitting to training data
    model.fit(X_train, y_train)

    # probability of converted = 1 only
    y_pred = model.predict_proba(X_val)[:, 1]

    # classify customers as converted if predicted probability is at least 50%
    is_converted = (y_pred >= 0.5)

    # calculate how often the predicted converted matches the actual value
    acc = (y_val == is_converted).mean()
    difference = accuracy - acc

    print(
        f"Without {e:25s} "
        f"accuracy = {accuracy:.5f}, "
        f"difference = {difference:.5f}"
    )

Without lead_source               accuracy = 0.64500, difference = 0.00300
Without number_of_courses_viewed  accuracy = 0.64500, difference = 0.00200
Without interaction_count         accuracy = 0.64500, difference = 0.04400


#### Q6. Which of these C leads to the best accuracy on the validation set?

In [18]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:

    # create a dict vectorizer (do not use sparse matrices)
    dv = DictVectorizer(sparse=False)

    # fit data to vectorizer and transform into one-hot-encoded
    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)

    # fit trains the model from scratch by fitting to training data
    model.fit(X_train, y_train)

    # probability of converted = 1 only
    y_pred = model.predict_proba(X_val)[:, 1]

    # classify customers as converted if predicted probability is at least 50%
    is_converted = (y_pred >= 0.5)

    # calculate how often the predicted converted matches the actual value
    acc = (y_val == is_converted).mean()

    print(f"accuracy = {acc:.3f}, c = {c}")

accuracy = 0.598, c = 1e-06
accuracy = 0.598, c = 1e-05
accuracy = 0.613, c = 0.0001
accuracy = 0.645, c = 0.001
